# DATATHON 2026 — Part 3: Sales Forecasting

**Strategy: Hybrid Prophet + LightGBM**

1. Prophet captures macro trend + yearly/weekly seasonality
2. LightGBM learns residual patterns from lag, rolling, calendar, and web-traffic features
3. Test predictions are filled iteratively (no leakage)
4. SHAP values printed for the technical report

**Output:** `submission.csv` (same folder)

## 0. Imports & Config

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)

DATA_DIR   = "dataset/"          # ← change if needed
OUT_FILE   = "submission.csv"
VAL_START  = "2021-01-01"        # time-based validation cut

# Auto-detect data directory
for candidate in ["dataset/", ".", "/mnt/user-data/uploads/"]:
    if os.path.exists(os.path.join(candidate, "sales.csv")):
        DATA_DIR = candidate
        break

print(f"[INFO] Data directory : {DATA_DIR}")
print(f"[INFO] Output file    : {OUT_FILE}")

## 1. Load Data

In [ ]:
sales  = pd.read_csv(os.path.join(DATA_DIR, "sales.csv"),             parse_dates=["Date"])
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), parse_dates=["Date"])
web    = pd.read_csv(os.path.join(DATA_DIR, "web_traffic.csv"),        parse_dates=["date"])

print(f"  sales.csv        : {sales.shape[0]:,} rows  | {sales['Date'].min().date()} → {sales['Date'].max().date()}")
print(f"  sample_sub.csv   : {sample.shape[0]:,} rows  | {sample['Date'].min().date()} → {sample['Date'].max().date()}")
print(f"  web_traffic.csv  : {web.shape[0]:,} rows")

## 2. Aggregate Web Traffic to Daily

In [ ]:
web_daily = (
    web.groupby("date")
    .agg(
        sessions             = ("sessions",                "sum"),
        unique_visitors      = ("unique_visitors",          "sum"),
        page_views           = ("page_views",               "sum"),
        bounce_rate          = ("bounce_rate",              "mean"),
        avg_session_duration = ("avg_session_duration_sec", "mean"),
    )
    .reset_index()
    .rename(columns={"date": "Date"})
)

print(f"  Web daily rows   : {web_daily.shape[0]:,}")
print(f"  Coverage         : {web_daily['Date'].min().date()} → {web_daily['Date'].max().date()}")
web_daily.head()

## 3. Prophet: Trend + Seasonality Features

In [ ]:
def fit_prophet(series_df, name="series", cp=0.1, sp=10):
    m = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = True,
        daily_seasonality       = False,
        changepoint_prior_scale = cp,
        seasonality_prior_scale = sp,
    )
    m.fit(series_df)
    print(f"  Prophet [{name}] fitted — best changepoints detected automatically")
    return m

m_rev  = fit_prophet(
    sales[["Date", "Revenue"]].rename(columns={"Date": "ds", "Revenue": "y"}),
    name="Revenue",
)
m_cogs = fit_prophet(
    sales[["Date", "COGS"]].rename(columns={"Date": "ds", "COGS": "y"}),
    name="COGS",
)

In [ ]:
# Predict prophet components for ALL dates (train + test)
all_ds = (
    pd.concat([sales[["Date"]], sample[["Date"]]])
    .drop_duplicates()
    .rename(columns={"Date": "ds"})
    .sort_values("ds")
    .reset_index(drop=True)
)

fc_rev = m_rev.predict(all_ds)[["ds", "yhat", "trend", "yearly", "weekly"]].rename(
    columns={"ds": "Date", "yhat": "p_rev", "trend": "p_trend",
             "yearly": "p_yearly", "weekly": "p_weekly"}
)
fc_cogs = m_cogs.predict(all_ds)[["ds", "yhat"]].rename(
    columns={"ds": "Date", "yhat": "p_cogs"}
)

print(f"  Prophet inference done for {len(all_ds):,} dates")

## 4. Feature Engineering

In [ ]:
def add_calendar_features(df):
    df = df.copy()
    df["year"]       = df["Date"].dt.year
    df["month"]      = df["Date"].dt.month
    df["day"]        = df["Date"].dt.day
    df["dow"]        = df["Date"].dt.dayofweek          # 0=Mon … 6=Sun
    df["doy"]        = df["Date"].dt.dayofyear
    df["woy"]        = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"]    = df["Date"].dt.quarter
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    # Vietnamese Tet: Jan–Feb first 15 days (approximate lunar new year window)
    df["is_tet"]     = ((df["month"].isin([1, 2])) & (df["day"] <= 15)).astype(int)
    # Year-end sales spike
    df["is_yearend"] = ((df["month"] == 12) & (df["day"] >= 20)).astype(int)
    # Mid-year sale (June–July)
    df["is_midyear"] = ((df["month"].isin([6, 7])) & (df["day"] <= 10)).astype(int)
    return df

# Build full timeline (train + test) for proper lag computation
full = (
    pd.concat([
        sales[["Date", "Revenue", "COGS"]],
        sample[["Date"]].assign(Revenue=np.nan, COGS=np.nan),
    ])
    .sort_values("Date")
    .reset_index(drop=True)
)

full = add_calendar_features(full)
full = full.merge(web_daily, on="Date", how="left")
full = full.merge(fc_rev,   on="Date", how="left")
full = full.merge(fc_cogs,  on="Date", how="left")

In [ ]:
# Lag features (only valid for known Revenue — test rows get NaN here,
# filled iteratively during prediction)
LAG_DAYS  = [7, 14, 21, 28, 365]
ROLL_DAYS = [7, 14, 30]

for lag in LAG_DAYS:
    full[f"lag_{lag}"] = full["Revenue"].shift(lag)
for win in ROLL_DAYS:
    full[f"roll_{win}"] = full["Revenue"].shift(1).rolling(win).mean()

FEATURES = [
    # Calendar
    "year", "month", "day", "dow", "doy", "woy", "quarter",
    "is_weekend", "is_tet", "is_yearend", "is_midyear",
    # Prophet components
    "p_rev", "p_trend", "p_yearly", "p_weekly", "p_cogs",
    # Web traffic
    "sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration",
    # Lag / rolling
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_365",
    "roll_7", "roll_14", "roll_30",
]

train_df = full[full["Revenue"].notna()].dropna(subset=["lag_365"]).copy()
test_df  = full[full["Revenue"].isna()].copy()

print(f"  Train rows (after lag warmup) : {len(train_df):,}")
print(f"  Test  rows                    : {len(test_df):,}")
print(f"  Feature count                 : {len(FEATURES)}")
print(f"  Features: {FEATURES}")

## 5. Time-based Cross-Validation

In [ ]:
trn = train_df[train_df["Date"] <  VAL_START]
val = train_df[train_df["Date"] >= VAL_START]
print(f"  Train period : {trn['Date'].min().date()} → {trn['Date'].max().date()}  ({len(trn):,} rows)")
print(f"  Val   period : {val['Date'].min().date()} → {val['Date'].max().date()}  ({len(val):,} rows)")

LGB_BASE = dict(
    learning_rate     = 0.02,
    num_leaves        = 63,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_estimators      = 3000,
)

def fit_lgb(X_tr, y_tr, X_va, y_va, label=""):
    m = lgb.LGBMRegressor(**LGB_BASE)
    m.fit(
        X_tr, y_tr,
        eval_set   = [(X_va, y_va)],
        callbacks  = [lgb.early_stopping(100, verbose=False),
                      lgb.log_evaluation(False)],
    )
    preds = m.predict(X_va)
    mae   = mean_absolute_error(y_va, preds)
    rmse  = np.sqrt(mean_squared_error(y_va, preds))
    r2    = r2_score(y_va, preds)
    print(f"  [{label}] best_iter={m.best_iteration_}")
    print(f"  [{label}] Val MAE  = {mae:>12,.0f}")
    print(f"  [{label}] Val RMSE = {rmse:>12,.0f}")
    print(f"  [{label}] Val R²   = {r2:.4f}")
    return m, preds

In [ ]:
print("Training Revenue model...")
m_rev_val,  vp_rev  = fit_lgb(trn[FEATURES], trn["Revenue"],
                               val[FEATURES], val["Revenue"],  label="Revenue")

In [ ]:
print("Training COGS model...")
m_cogs_val, vp_cogs = fit_lgb(trn[FEATURES], trn["COGS"],
                               val[FEATURES], val["COGS"],     label="COGS")

## 6. Retrain on Full Training Data

In [ ]:
def retrain_full(train_full, target, n_iter, label=""):
    params = {**LGB_BASE, "n_estimators": n_iter}
    m = lgb.LGBMRegressor(**params)
    m.fit(train_full[FEATURES], train_full[target],
          callbacks=[lgb.log_evaluation(False)])
    print(f"  [{label}] retrained with n_estimators={n_iter}")
    return m

m_rev_final  = retrain_full(train_df, "Revenue", m_rev_val.best_iteration_,  "Revenue")
m_cogs_final = retrain_full(train_df, "COGS",    m_cogs_val.best_iteration_, "COGS")

## 7. Iterative Test Prediction (No Leakage)

In [ ]:
rev_series  = full["Revenue"].copy()
cogs_series = full["COGS"].copy()
test_indices = test_df.index.tolist()

print(f"  Predicting {len(test_indices)} days iteratively...")

for pos, idx in enumerate(test_indices):
    row = full.loc[idx].copy()

    # Recompute lag features using already-predicted values
    for lag in LAG_DAYS:
        src = idx - lag
        if src >= 0 and not pd.isna(rev_series.iloc[src]):
            row[f"lag_{lag}"] = rev_series.iloc[src]

    # Recompute rolling means
    for win in ROLL_DAYS:
        vals = [
            rev_series.iloc[idx - j]
            for j in range(1, win + 1)
            if (idx - j) >= 0 and not pd.isna(rev_series.iloc[idx - j])
        ]
        if vals:
            row[f"roll_{win}"] = np.mean(vals)

    X = pd.DataFrame([row[FEATURES]])
    pred_rev  = float(m_rev_final.predict(X)[0])
    pred_cogs = float(m_cogs_final.predict(X)[0])

    rev_series.iloc[idx]  = pred_rev
    cogs_series.iloc[idx] = pred_cogs

    if (pos + 1) % 100 == 0 or pos == 0:
        print(f"    {pos+1}/{len(test_indices)} — {full.loc[idx,'Date'].date()} "
              f"→ Rev={pred_rev:>12,.0f}  COGS={pred_cogs:>12,.0f}")

print("  Done.")

## 8. Final Validation Metrics Summary

In [ ]:
print(f"  {'Metric':<10} {'Revenue':>15} {'COGS':>15}")
print(f"  {'-'*42}")
print(f"  {'MAE':<10} {mean_absolute_error(val['Revenue'], vp_rev):>15,.0f} "
      f"{mean_absolute_error(val['COGS'], vp_cogs):>15,.0f}")
print(f"  {'RMSE':<10} {np.sqrt(mean_squared_error(val['Revenue'], vp_rev)):>15,.0f} "
      f"{np.sqrt(mean_squared_error(val['COGS'], vp_cogs)):>15,.0f}")
print(f"  {'R²':<10} {r2_score(val['Revenue'], vp_rev):>15.4f} "
      f"{r2_score(val['COGS'], vp_cogs):>15.4f}")

## 9. SHAP Feature Importance

In [ ]:
explainer   = shap.TreeExplainer(m_rev_final)
shap_sample = train_df[FEATURES].sample(min(500, len(train_df)), random_state=42)
shap_values = explainer.shap_values(shap_sample)
mean_abs    = np.abs(shap_values).mean(axis=0)
importance  = (
    pd.DataFrame({"feature": FEATURES, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

print("  Top 15 Revenue drivers (SHAP):")
print(f"  {'Rank':<5} {'Feature':<30} {'Mean |SHAP|':>14}")
print(f"  {'-'*52}")
for i, row in importance.head(15).iterrows():
    print(f"  {i+1:<5} {row['feature']:<30} {row['mean_abs_shap']:>14,.0f}")

In [ ]:
# Business interpretation
business_map = {
    "p_rev":      "Prophet trend/seasonality — long-run revenue trajectory",
    "p_trend":    "Structural trend component — growth/decline over years",
    "p_yearly":   "Yearly seasonality — peak/trough months",
    "p_weekly":   "Day-of-week seasonality — weekend vs weekday pattern",
    "lag_365":    "Same day last year — strong year-over-year pattern",
    "lag_7":      "Revenue 7 days ago — short-term momentum",
    "roll_30":    "30-day rolling average — medium-term baseline",
    "sessions":   "Website sessions — demand signal leading indicator",
    "page_views": "Page views — browsing intent proxy",
    "is_tet":     "Tet holiday flag — Vietnamese New Year sales surge",
    "is_yearend": "Year-end flag — December shopping season",
    "month":      "Month of year — captures seasonal promotions",
    "year":       "Year — encodes long-term structural trend",
}

print("Business interpretation of top 10 features:")
for feat in importance.head(10)["feature"]:
    if feat in business_map:
        print(f"  → {feat:<20}: {business_map[feat]}")

## 10. Plots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Actual vs Predicted on validation
axes[0].plot(val["Date"], val["Revenue"],  lw=1.0, label="Actual",    color="#2196F3")
axes[0].plot(val["Date"], vp_rev,          lw=1.0, label="Predicted", color="#FF5722", linestyle="--")
axes[0].set_title("Revenue — Actual vs Predicted (Validation 2021–2022)", fontsize=13)
axes[0].set_ylabel("Revenue (VND)")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 2: Test period predictions
test_dates = full.loc[test_indices, "Date"]
test_rev   = rev_series.iloc[test_indices].values
axes[1].plot(test_dates, test_rev, lw=1.0, color="#4CAF50", label="Forecast")
axes[1].set_title("Revenue Forecast — Test Period (2023-01-01 → 2024-07-01)", fontsize=13)
axes[1].set_ylabel("Revenue (VND)")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 3: SHAP bar chart (top 15)
top15 = importance.head(15).sort_values("mean_abs_shap")
axes[2].barh(top15["feature"], top15["mean_abs_shap"], color="#9C27B0")
axes[2].set_title("Top 15 Features — Mean |SHAP| Value (Revenue model)", fontsize=13)
axes[2].set_xlabel("Mean |SHAP value|")
axes[2].grid(alpha=0.3, axis="x")

plt.tight_layout()
plot_path = OUT_FILE.replace(".csv", "_plots.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved plots → {plot_path}")

## 11. Export Submission

In [ ]:
submission = sample[["Date"]].copy()
submission["Revenue"] = np.round(rev_series.iloc[test_indices].values, 2)
submission["COGS"]    = np.round(cogs_series.iloc[test_indices].values, 2)
submission["Date"]    = submission["Date"].dt.strftime("%Y-%m-%d")

submission.to_csv(OUT_FILE, index=False)
print(f"  ✓ Saved {len(submission):,} rows → {OUT_FILE}")
print("\n  Preview:")
submission.head(10)

---
### ✅ ALL DONE — Ready to submit to Kaggle!